# 典型协同场景：稀疏索引选择类算子

协同模式最大的用武之地：**先"选择"再"计算"** 的算子族——门控、top-k 检索、稀疏注意力。第 11 章的 Gated-KV SDPA 就是这一族的矩阵级成员。


## 1. 模式抽象：Select-then-Compute

```text
输入：候选集 D（大、规整） + 打分/索引信号 S（小或不规则）
阶段一（选择，SIMT 倾向）：按 S 从 D 中选出子集 D'（索引离散、可能竞争）
阶段二（计算，SIMD/Cube 倾向）：对 D' 做规整计算（matmul/逐元素）
交接：D' 或其索引表 + 元数据（选中数量等）
```

选择段的特征（第 7 章已建立）：读地址运行时决定（Gather 型）、写目标竞争（原子仲裁型）、块内协作（直方图/计数）。计算段的特征：选中数据组织成规整块后走 Tile/Cube 接口。


## 2. 实例一：门控 FFN（Mixture-of-Experts 雏形）

`y = expert_{g(i)}(x_i)`——每个样本 i 由门控分数 g 决定送去哪个专家（列子集）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">段</th><th style="text-align:left;">归属</th><th style="text-align:left;">理由</th></tr>
<tr><td>门控打分 g(i)（小 matmul）</td><td>SIMD/Cube</td><td>规整矩阵乘</td></tr>
<tr><td>argmax/top-k 选专家</td><td>SIMT</td><td>逐样本比较 + 竞争仲裁（先明确 TopK 次序；第 7 章的 Scatter 是另一种仲裁协议）</td></tr>
<tr><td>按专家分组路由</td><td>SIMT（scatter 累积计数）</td><td>写目标运行时决定 + 直方图式计数</td></tr>
<tr><td>分组后的批量 GEMM</td><td>Cube</td><td>各组内规整</td></tr>
</table>

真实 MoE 算子的分组 GEMM 在变长组上还有大量工程细节——此处建立的是**计算图分解直觉**。


## 3. 实例二：稀疏 KV 选择的注意力（第 11 章预习）

`Attention(Q, K_selected, V_selected)`——KV 块按门控分数选择：

- 打分（Q·k 或轻量索引网络）→ 规整计算（Cube/SIMD）；
- **块选择**（top-k + 竞争仲裁）→ SIMT（第 7 章实践三的模式放大到块粒度）；
- 选中块的注意力计算 → 回到 SDPA 主干（第 10 章）。

注意选择粒度是**块**（如 [128, 64] 的 KV 块）而非单个元素——块粒度让"选择后计算"天然规整，这正是稀疏注意力能用协同模式高效实现的原因。


## 4. 选择段的替代路径：Tile 级接口 vs SIMT 仲裁

本节一直默认"选择段用 SIMT 门槛仲裁"，但 03.03 扩展接口面显示 Pro 还有 **Tile 级选择接口**。二者怎么选？正式对照：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">Tile 级（gather 比较模式 / argmax / sort32）</th><th style="text-align:left;">SIMT 门槛仲裁</th></tr>
<tr><td>数据位置</td><td>片上 Tile（需先 load 进 UB）</td><td>GM Tensor 直读（tensor[i,j]）</td></tr>
<tr><td>选择形态</td><td>阈值筛选（gather 比较模式写索引+计数）、极值索引（argmax）、定长排序（sort32）</td><td>任意门槛规则 + 竞争取号，可承载复杂仲裁（同分破平、名额上限）</td></tr>
<tr><td>适用</td><td>候选集能装进 UB、规则简单（阈值/top-1/定长 top-k）</td><td>候选跨 GM、规则复杂、变长输出</td></tr>
<tr><td>工程代价</td><td>接口即语义，代码短</td><td>须实现真正的选择算法与同步协议（第 9 章实践三采用确定性排名）</td></tr>
</table>

**选型结论**：教学主线保留 SIMT 仲裁（覆盖最广、最能训练"竞争仲裁"这一第 11 章核心技能）；但**简单规则 + 候选在片上**的场景（如 top-1 门控、阈值初筛），优先考虑 tile 级接口——可作为实现候选，具体收益需基准验证，第 11 章延伸阅读对照 gym 算子时可留意。


## 5. 交接的工程细节

1. **选中数量不确定**：输出区按上界分配（D' 大小 ≤ D），实际数量作为元数据传给计算段（变长 → 固定 rank + count 的静态 Shape 模拟，第 7 章能力边界的应对）；
2. **紧凑化**：选中项 scatter 到输出区头部（索引连续化），让计算段按规整块处理；
3. **元数据同步**：索引、计数和对应数据都必须发布完成后才允许消费。原子计数器取号只分配槽位，不表示槽内数据已写完；GM 直写不自动受 TileGroup mutex 保护。跨 Block 可先用分 Kernel 同 Stream 的阶段边界，同 Kernel 需显式证明所有参与核的发布/等待协议。


### 交接表应写出有效长度和原始位置

假设来源行 [0,2,5] 命中，原子取号可能得到紧凑顺序 [5,0,2]。count=3 只说明有效区长度；source_ids=[5,0,2] 才定义 packed[r] 对应哪一行。后处理应只读 packed[:3]，验收按 source_ids 取 Golden，不能假设恢复为来源升序。

还须覆盖 count=0：此时不启动需要正有效形状的 Tile 处理，输出有效区为空；count=1024 时则恰好占满上界，不应额外写第 1025 行。参考实现通过完整生产 Kernel 的结束和 Host 同步读取 count，建立清楚的阶段边界。后续同 Kernel 改造必须另证 GM 发布、缓存和所有参与核的可见性。


## 6. 课后练习

1. （填空）Select-then-Compute 的选择段特征：读地址 ______、写目标可能 ______；计算段特征：选中数据组织成 ______ 后走批量接口。
2. （实践）把"Top-k 门控 FFN"画成五步设计流程的表格（段/归属/交接载体/同步点），标注哪一步复用了第 7 章的哪个模式。

In [ ]:
!cat ./answer/09.03_answer.txt

## 7. 本节小结

- **Select-then-Compute** 是协同的最大应用族：选择（SIMT：Gather/仲裁/计数）+ 计算（SIMD/Cube：规整批量）；
- 块粒度选择让计算段天然规整——稀疏注意力的可行性来源；
- 交接工程三件套：**上界分配 + 紧凑化 + 元数据同步**。